# Online Retail II - Association Rule Mining Preprocessing

This notebook loads the Online Retail II dataset and transforms it into transaction format for association rule mining algorithms.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# Add src directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'src'))

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Load Raw Data

In [2]:
import urllib.request

# Set up paths
raw_data_path = Path.cwd().parent / 'data' / 'raw' / 'online_retail_II.xlsx'
processed_data_path = Path.cwd().parent / 'data' / 'processed'

# Create directories if they don't exist
raw_data_path.parent.mkdir(parents=True, exist_ok=True)
processed_data_path.mkdir(parents=True, exist_ok=True)

# Download if not exists
if not raw_data_path.exists():
    print(f"File not found at {raw_data_path}")
    print("Downloading from UCI Machine Learning Repository...")
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00502/online_retail_II.xlsx"
    try:
        urllib.request.urlretrieve(url, raw_data_path)
        print(f"✓ Downloaded successfully to {raw_data_path}\n")
    except Exception as e:
        print(f"✗ Download failed: {e}")
        raise

print(f"Loading data from: {raw_data_path}")

# Load the dataset
df = pd.read_excel(raw_data_path)

print(f"\nDataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst few rows:")
df.head()

File not found at /Users/nangvuong/Desktop/association-rule-project/data/raw/online_retail_II.xlsx
✓ Downloaded successfully to /Users/nangvuong/Desktop/association-rule-project/data/raw/online_retail_II.xlsx

Loading data from: /Users/nangvuong/Desktop/association-rule-project/data/raw/online_retail_II.xlsx

Dataset loaded successfully!
Shape: (525461, 8)

Column names and types:
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

First few rows:


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## 3. Explore Dataset Structure

In [3]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())
print(f"\nPercentage of missing values:")
print((df.isnull().sum() / len(df) * 100).round(2))

# Basic statistics
print(f"\n\nBasic Statistics:")
print(f"Total transactions: {df.shape[0]}")
print(f"Total unique invoices: {df['Invoice'].nunique()}")
print(f"Total unique products: {df['StockCode'].nunique()}")
print(f"Total unique customers: {df['Customer ID'].nunique()}")
print(f"Date range: {df['InvoiceDate'].min()} to {df['InvoiceDate'].max()}")

# Check quantity distribution
print(f"\nQuantity statistics:")
print(df['Quantity'].describe())

# Check price distribution
print(f"\nPrice statistics:")
print(df['Price'].describe())

Missing values:
Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

Percentage of missing values:
Invoice         0.00
StockCode       0.00
Description     0.56
Quantity        0.00
InvoiceDate     0.00
Price           0.00
Customer ID    20.54
Country         0.00
dtype: float64


Basic Statistics:
Total transactions: 525461
Total unique invoices: 28816
Total unique products: 4632
Total unique customers: 4383
Date range: 2009-12-01 07:45:00 to 2010-12-09 20:01:00

Quantity statistics:
count    525461.000000
mean         10.337667
std         107.424110
min       -9600.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       19152.000000
Name: Quantity, dtype: float64

Price statistics:
count    525461.000000
mean          4.688834
std         146.126914
min      -53594.360000
25%           1.250000
50%           2.100000

## 4. Data Cleaning

In [4]:
print(f"Original dataset size: {len(df)}")

# Create a copy for cleaning
df_clean = df.copy()

# Step 1: Remove rows with missing Invoice, Description, or Customer ID
print(f"\nStep 1: Removing rows with missing Invoice, Description, or Customer ID...")
df_clean = df_clean.dropna(subset=['Invoice', 'Description', 'Customer ID'])
print(f"Removed {len(df) - len(df_clean)} rows")

# Step 2: Remove cancelled transactions (Invoice starting with 'C' or negative quantities)
print(f"\nStep 2: Removing cancelled transactions...")
initial_len = len(df_clean)
df_clean = df_clean[~df_clean['Invoice'].astype(str).str.startswith('C')]
df_clean = df_clean[df_clean['Quantity'] > 0]
print(f"Removed {initial_len - len(df_clean)} rows")

# Step 3: Remove non-positive prices
print(f"\nStep 3: Removing non-positive prices...")
initial_len = len(df_clean)
df_clean = df_clean[df_clean['Price'] > 0]
print(f"Removed {initial_len - len(df_clean)} rows")

# Step 4: Remove duplicates (keeping first occurrence)
print(f"\nStep 4: Removing duplicates...")
initial_len = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['Invoice', 'StockCode', 'Customer ID'], keep='first')
print(f"Removed {initial_len - len(df_clean)} rows")

# Step 5: Remove rows with empty or whitespace-only descriptions
print(f"\nStep 5: Cleaning descriptions...")
initial_len = len(df_clean)
df_clean['Description'] = df_clean['Description'].str.strip()
df_clean = df_clean[df_clean['Description'].str.len() > 0]
print(f"Removed {initial_len - len(df_clean)} rows")

print(f"\nFinal cleaned dataset size: {len(df_clean)}")
print(f"Data cleaning completed. Removed {len(df) - len(df_clean)} rows in total ({(len(df) - len(df_clean)) / len(df) * 100:.2f}%)")

# Display cleaned data info
print(f"\nCleaned data info:")
print(f"Total transactions: {len(df_clean)}")
print(f"Unique invoices: {df_clean['Invoice'].nunique()}")
print(f"Unique products: {df_clean['StockCode'].nunique()}")
print(f"Unique customers: {df_clean['Customer ID'].nunique()}")

Original dataset size: 525461

Step 1: Removing rows with missing Invoice, Description, or Customer ID...
Removed 107927 rows

Step 2: Removing cancelled transactions...
Removed 9839 rows

Step 3: Removing non-positive prices...
Removed 31 rows

Step 4: Removing duplicates...
Removed 12704 rows

Step 5: Cleaning descriptions...
Removed 0 rows

Final cleaned dataset size: 394960
Data cleaning completed. Removed 130501 rows in total (24.84%)

Cleaned data info:
Total transactions: 394960
Unique invoices: 19213
Unique products: 4017
Unique customers: 4312


## 5. Filter Rare Items

In [5]:
# Calculate item frequency (support)
print("Calculating item frequency (support)...\n")

item_counts = df_clean['Description'].value_counts()
print(f"Total unique items before filtering: {len(item_counts)}")
print(f"\nItem frequency statistics:")
print(item_counts.describe())

# Set threshold for rare items
min_support_count = 50  # Items must appear at least 50 times
min_support_percentage = 0.1  # Items must appear in at least 0.1% of transactions
min_support_count_relative = int(len(df_clean) * (min_support_percentage / 100))

print(f"\n{'='*60}")
print(f"Rare Item Filtering Configuration:")
print(f"{'='*60}")
print(f"Total records: {len(df_clean)}")
print(f"Absolute threshold: {min_support_count} occurrences")
print(f"Relative threshold: {min_support_percentage}% = {min_support_count_relative} occurrences")

# Use the higher threshold
threshold = max(min_support_count, min_support_count_relative)
print(f"Using threshold: {threshold} occurrences")

# Identify rare and common items
rare_items = item_counts[item_counts < threshold].index.tolist()
common_items = item_counts[item_counts >= threshold].index.tolist()

print(f"\n{'='*60}")
print(f"Filtering Results:")
print(f"{'='*60}")
print(f"Rare items (frequency < {threshold}): {len(rare_items)}")
print(f"Common items (frequency >= {threshold}): {len(common_items)}")
print(f"Items to remove: {len(rare_items)} ({len(rare_items)/len(item_counts)*100:.2f}%)")

# Display top rare and common items
print(f"\nTop 10 most frequent items (KEEP):")
print(item_counts.head(10))

print(f"\nTop 10 rarest items (REMOVE):")
print(item_counts.tail(10))

# Filter the dataset - remove rows with rare items
print(f"\n\nRemoving rare items...")
initial_len = len(df_clean)
df_clean = df_clean[df_clean['Description'].isin(common_items)]
removed_rows = initial_len - len(df_clean)

print(f"Records removed due to rare items: {removed_rows}")
print(f"Percentage removed: {removed_rows/initial_len*100:.2f}%")

print(f"\n{'='*60}")
print(f"Dataset after removing rare items:")
print(f"{'='*60}")
print(f"Total records: {len(df_clean)}")
print(f"Unique invoices: {df_clean['Invoice'].nunique()}")
print(f"Unique products: {df_clean['StockCode'].nunique()}")
print(f"Unique customers: {df_clean['Customer ID'].nunique()}")
print(f"Unique items (descriptions): {df_clean['Description'].nunique()}")
print(f"{'='*60}")

Calculating item frequency (support)...

Total unique items before filtering: 4406

Item frequency statistics:
count    4406.000000
mean       89.641398
std       148.182652
min         1.000000
25%        10.000000
50%        34.500000
75%       106.750000
max      3021.000000
Name: count, dtype: float64

Rare Item Filtering Configuration:
Total records: 394960
Absolute threshold: 50 occurrences
Relative threshold: 0.1% = 394 occurrences
Using threshold: 394 occurrences

Filtering Results:
Rare items (frequency < 394): 4229
Common items (frequency >= 394): 177
Items to remove: 4229 (95.98%)

Top 10 most frequent items (KEEP):
Description
WHITE HANGING HEART T-LIGHT HOLDER    3021
REGENCY CAKESTAND 3 TIER              1687
STRAWBERRY CERAMIC TRINKET BOX        1335
ASSORTED COLOUR BIRD ORNAMENT         1333
HOME BUILDING BLOCK WORD              1174
PACK OF 72 RETRO SPOT CAKE CASES      1160
60 TEATIME FAIRY CAKE CASES           1136
JUMBO BAG RED RETROSPOT               1060
LUNCH BAG

## 6. Transform to Transaction Format

In [6]:
# Method 1: Group by Invoice (transaction-level)
# Each invoice represents a transaction/basket
print("Creating transaction format by grouping by Invoice...")

transactions = df_clean.groupby('Invoice')['Description'].apply(list).reset_index()
transactions.columns = ['Invoice', 'Items']

print(f"Total transactions (invoices): {len(transactions)}")
print(f"Average items per transaction: {transactions['Items'].apply(len).mean():.2f}")
print(f"Min items per transaction: {transactions['Items'].apply(len).min()}")
print(f"Max items per transaction: {transactions['Items'].apply(len).max()}")

# Display sample transactions
print(f"\nSample transactions:")
for i in range(min(5, len(transactions))):
    print(f"Transaction {i+1} ({len(transactions.iloc[i]['Items'])} items): {transactions.iloc[i]['Items'][:5]}...")

# Filter transactions with at least 2 items for meaningful association rules
print(f"\n\nFiltering transactions with at least 2 items...")
transactions_filtered = transactions[transactions['Items'].apply(len) >= 2].copy()
print(f"Transactions with 2+ items: {len(transactions_filtered)}")

# Create final transaction list (list of itemsets)
transaction_list = [set(items) for items in transactions_filtered['Items']]

print(f"\nSample itemsets:")
for i in range(min(3, len(transaction_list))):
    print(f"  {transaction_list[i]}")

Creating transaction format by grouping by Invoice...
Total transactions (invoices): 16347
Average items per transaction: 6.75
Min items per transaction: 1
Max items per transaction: 59

Sample transactions:
Transaction 1 (1 items): ['STRAWBERRY CERAMIC TRINKET BOX']...
Transaction 2 (6 items): ['LOVE BUILDING BLOCK WORD', 'HOME BUILDING BLOCK WORD', 'ASSORTED COLOUR BIRD ORNAMENT', 'HEART IVORY TRELLIS LARGE', 'HEART FILIGREE DOVE LARGE']...
Transaction 3 (5 items): ['HANGING HEART ZINC T-LIGHT HOLDER', 'PINK BLUE FELT CRAFT TRINKET BOX', 'FELTCRAFT DOLL ROSIE', 'SCOTTIE DOG HOT WATER BOTTLE', 'CHOCOLATE HOT WATER BOTTLE']...
Transaction 4 (1 items): ['JUMBO BAG TOYS']...
Transaction 5 (4 items): ['BAKING SET 9 PIECE RETROSPOT', 'RETRO SPOT TEA SET CERAMIC 11 PC', 'RED TOADSTOOL LED NIGHT LIGHT', 'POSTAGE']...


Filtering transactions with at least 2 items...
Transactions with 2+ items: 14070

Sample itemsets:
  {'HEART IVORY TRELLIS LARGE', 'HEART FILIGREE DOVE LARGE', 'HOME BUILDING

## 7. One-Hot Encoding

In [7]:
# Create one-hot encoded matrices for transactions
print("Creating one-hot encoded matrices...\n")

# Get all unique items
all_items = sorted(list(set().union(*transaction_list)))
print(f"Total unique items: {len(all_items)}")

# Create one-hot encoded matrix for invoice-based transactions
def create_onehot_matrix(transactions, all_items):
    """
    Create a binary matrix where rows are transactions and columns are items
    Value is 1 if item is in transaction, 0 otherwise
    """
    matrix = np.zeros((len(transactions), len(all_items)), dtype=int)
    item_to_idx = {item: idx for idx, item in enumerate(all_items)}
    
    for trans_idx, transaction in enumerate(transactions):
        for item in transaction:
            item_idx = item_to_idx[item]
            matrix[trans_idx, item_idx] = 1
    
    return matrix, item_to_idx

# Create one-hot matrices
print("Creating one-hot matrix for invoice-based transactions...")
onehot_matrix, item_to_idx = create_onehot_matrix(transaction_list, all_items)
print(f"Shape: {onehot_matrix.shape}")
print(f"Sparsity: {(onehot_matrix == 0).sum() / onehot_matrix.size * 100:.2f}% zeros")

# Create DataFrame for easy viewing and saving
onehot_df = pd.DataFrame(
    onehot_matrix,
    columns=all_items
)

print(f"\nOne-hot encoded dataframe (first 5 transactions):")
print(onehot_df.head())

# Statistics
print(f"\nItem frequency statistics:")
item_freq = onehot_df.sum(axis=0)
print(f"Min frequency: {item_freq.min()}")
print(f"Max frequency: {item_freq.max()}")
print(f"Mean frequency: {item_freq.mean():.2f}")
print(f"Median frequency: {item_freq.median():.2f}")

# Display most and least frequent items
print(f"\nTop 10 most frequent items:")
print(item_freq.nlargest(10))

print(f"\nTop 10 least frequent items:")
print(item_freq.nsmallest(10))

Creating one-hot encoded matrices...

Total unique items: 177
Creating one-hot matrix for invoice-based transactions...
Shape: (14070, 177)
Sparsity: 95.66% zeros

One-hot encoded dataframe (first 5 transactions):
   12 PENCILS SMALL TUBE SKULL  3 HEARTS HANGING DECORATION RUSTIC  \
0                            0                                   0   
1                            0                                   0   
2                            0                                   0   
3                            0                                   0   
4                            0                                   0   

   3 STRIPEY MICE FELTCRAFT  6 RIBBONS RUSTIC CHARM  \
0                         0                       0   
1                         0                       0   
2                         0                       0   
3                         0                       0   
4                         0                       0   

   60 TEATIME FAIRY CAKE CASES  72

## 8. Save Processed Data

In [8]:
# Save processed data
print("Saving processed data...\n")

# Save 1: Cleaned data (CSV)
cleaned_csv_path = processed_data_path / 'cleaned_data.csv'
df_clean.to_csv(cleaned_csv_path, index=False)
print(f"✓ Saved cleaned data: {cleaned_csv_path}")

# Save 2: Transactions (CSV)
transactions_csv_path = processed_data_path / 'transactions.csv'
transactions_filtered.to_csv(transactions_csv_path, index=False)
print(f"✓ Saved transactions: {transactions_csv_path}")

# Save 3: Transaction list (Pickle format)
transaction_list_pkl_path = processed_data_path / 'transaction_list.pkl'
with open(transaction_list_pkl_path, 'wb') as f:
    pickle.dump(transaction_list, f)
print(f"✓ Saved transaction list (pickle): {transaction_list_pkl_path}")

# Save 4: Transaction list as text file (one transaction per line)
transaction_text_path = processed_data_path / 'transactions.txt'
with open(transaction_text_path, 'w', encoding='utf-8') as f:
    for transaction in transaction_list:
        items = ','.join(sorted(transaction))
        f.write(items + '\n')
print(f"✓ Saved transaction list as text: {transaction_text_path}")

# Save 5: One-hot matrix (CSV)
onehot_csv_path = processed_data_path / 'onehot_matrix.csv'
onehot_df.to_csv(onehot_csv_path, index=False)
print(f"✓ Saved one-hot matrix (CSV): {onehot_csv_path}")

# Save 6: One-hot matrix (NumPy format)
onehot_npy_path = processed_data_path / 'onehot_matrix.npy'
np.save(onehot_npy_path, onehot_matrix)
print(f"✓ Saved one-hot matrix (NPY): {onehot_npy_path}")

# Save 7: Item mapping (which column index corresponds to which item)
item_mapping_path = processed_data_path / 'item_mapping.pkl'
with open(item_mapping_path, 'wb') as f:
    pickle.dump(item_to_idx, f)
print(f"✓ Saved item mapping (pickle): {item_mapping_path}")

# Save 8: Item mapping as CSV for easy reference
item_mapping_df = pd.DataFrame(
    list(item_to_idx.items()),
    columns=['Item', 'ColumnIndex']
).sort_values('ColumnIndex')
item_mapping_csv_path = processed_data_path / 'item_mapping.csv'
item_mapping_df.to_csv(item_mapping_csv_path, index=False)
print(f"✓ Saved item mapping (CSV): {item_mapping_csv_path}")

# Save 9: Summary statistics
summary_stats = {
    'Total transactions': len(transactions_filtered),
    'Total unique items': len(all_items),
    'Onehot matrix shape': f"{onehot_matrix.shape[0]} x {onehot_matrix.shape[1]}",
    'Avg items per transaction': f"{transactions_filtered['Items'].apply(len).mean():.2f}",
    'Min items per transaction': int(transactions_filtered['Items'].apply(len).min()),
    'Max items per transaction': int(transactions_filtered['Items'].apply(len).max()),
    'Matrix sparsity': f"{(onehot_matrix == 0).sum() / onehot_matrix.size * 100:.2f}%",
}

summary_df = pd.DataFrame(list(summary_stats.items()), columns=['Metric', 'Value'])
summary_csv_path = processed_data_path / 'processing_summary.csv'
summary_df.to_csv(summary_csv_path, index=False)
print(f"✓ Saved summary statistics: {summary_csv_path}")

print(f"\n{'='*60}")
print(f"Processing Summary:")
print(f"{'='*60}")
for metric, value in summary_stats.items():
    print(f"{metric:<40} {value}")
print(f"{'='*60}")
print(f"\nAll files saved to: {processed_data_path}")

Saving processed data...

✓ Saved cleaned data: /Users/nangvuong/Desktop/association-rule-project/data/processed/cleaned_data.csv
✓ Saved transactions: /Users/nangvuong/Desktop/association-rule-project/data/processed/transactions.csv
✓ Saved transaction list (pickle): /Users/nangvuong/Desktop/association-rule-project/data/processed/transaction_list.pkl
✓ Saved transaction list as text: /Users/nangvuong/Desktop/association-rule-project/data/processed/transactions.txt
✓ Saved one-hot matrix (CSV): /Users/nangvuong/Desktop/association-rule-project/data/processed/onehot_matrix.csv
✓ Saved one-hot matrix (NPY): /Users/nangvuong/Desktop/association-rule-project/data/processed/onehot_matrix.npy
✓ Saved item mapping (pickle): /Users/nangvuong/Desktop/association-rule-project/data/processed/item_mapping.pkl
✓ Saved item mapping (CSV): /Users/nangvuong/Desktop/association-rule-project/data/processed/item_mapping.csv
✓ Saved summary statistics: /Users/nangvuong/Desktop/association-rule-project/da